# Лабораторная работа №11: Alignment VLM по предпочтениям

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Собрать preference pairs без length/style shortcut.
- Провести LoRA preference optimization на 2B fallback.
- Сравнить base/SFT/aligned по win rate и hallucinations.


## Теоретическая часть

Preference optimization использует пары chosen/rejected. DPO увеличивает относительную вероятность chosen относительно reference policy без отдельной reward model. Для VLM важно, чтобы изображения и prompts совпадали в обеих ветвях.


### Математическая постановка

$$\mathcal L_{DPO}=-\log\sigma\left(\beta[\log\tfrac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)}-\log\tfrac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}]\right).$$


### Материалы

- [DPO paper](https://arxiv.org/abs/2305.18290)
- [TRL цикл оптимизации DPO](https://huggingface.co/docs/trl/dpo_trainer)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Preference dataset является экспертным входом; пример с несуществующим cats.jpg не используется. Нужен адаптер lab 9 и GPU. При смене версии TRL сверьте мультимодальный цикл оптимизации DPO и зафиксируйте версии. `FAST_DEV_RUN=True` выполняет реальные два шага DPO, но не является evidence качества. Без заполненного слепого экспертного аудита итоговая метрика не создаётся.

In [ ]:
from pathlib import Path
import json, os, random, time
import numpy as np
SEED = 42
FAST_DEV_RUN = False  # True: проверка механики на малой выборке, НЕ отчётный эксперимент
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
def save_json(name, data):
    (ARTIFACTS/name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str))
from pathlib import Path
import json, hashlib
from datasets import load_dataset

SCIENCEQA_ID = 'derek-thomas/ScienceQA'
SCIENCEQA_REVISION = 'f18b0a70359ebfb41f658fd564208d0355b013f4'

def prepare_scienceqa(directory, smoke=False):
    """Fixed official splits, image-level deduplication, explicitly defined shifts."""
    directory = Path(directory).resolve(); directory.mkdir(parents=True, exist_ok=True)
    manifest = directory / ('manifest-smoke.jsonl' if smoke else 'manifest.jsonl')
    limits = {'train': 8 if smoke else 512, 'validation': 4 if smoke else 64,
              'id_test': 4 if smoke else 64, 'ood_test': 4 if smoke else 64,
              'general_test': 4 if smoke else 64}
    if manifest.exists():
        records = [json.loads(s) for s in manifest.read_text().splitlines()]
        if all(Path(r['image']).is_file() for r in records) and all(sum(r['split']==s for r in records)==n for s,n in limits.items()):
            return manifest
    records, seen, counts = [], set(), dict.fromkeys(limits, 0)
    for official in ('train', 'validation', 'test'):
        data = load_dataset(SCIENCEQA_ID, revision=SCIENCEQA_REVISION, split=official, streaming=True)
        for index, row in enumerate(data):
            image = row['image']
            if image is None: continue
            if official != 'test':
                if row['subject'] != 'natural science' or int(row['grade'].removeprefix('grade')) > 6: continue
                split = official
            elif row['subject'] != 'natural science': split = 'general_test'
            elif int(row['grade'].removeprefix('grade')) > 6: split = 'ood_test'
            else: split = 'id_test'
            if counts[split] >= limits[split]: continue
            image = image.convert('RGB')
            digest = hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
            if digest in seen: continue
            seen.add(digest)
            path = directory / f'{digest}.png'; image.save(path)
            answer = int(row['answer']); choices = row['choices']
            question = row['question']+'\n'+'\n'.join(f'{chr(65+i)}. {v}' for i,v in enumerate(choices))+'\nReturn only the letter of the correct option.'
            records.append({'id': f'scienceqa-{official}-{index}', 'split': split,
                            'image': str(path), 'question': question, 'answer': chr(65+answer),
                            'choices': choices, 'answer_index': answer, 'label_type': 'multiple_choice',
                            'license': 'MIT (code license, lupantech/ScienceQA)',
                            'source_url': f'https://huggingface.co/datasets/{SCIENCEQA_ID}/tree/{SCIENCEQA_REVISION}',
                            'subject':row['subject'], 'grade':row['grade'], 'image_sha256':digest})
            counts[split] += 1
            required = [official] if official != 'test' else ['id_test','ood_test','general_test']
            if all(counts[s] == limits[s] for s in required): break
    if counts != limits: raise RuntimeError(f'Insufficient licensed image examples: {counts}; expected {limits}')
    manifest.write_text('\n'.join(json.dumps(r,ensure_ascii=False) for r in records)+'\n')
    (directory/'ATTRIBUTION.md').write_text('ScienceQA — Lu et al., NeurIPS 2022. Dataset distribution: '+SCIENCEQA_ID+'\nRevision: '+SCIENCEQA_REVISION+'\nDataset-card license: CC-BY-SA-4.0, https://creativecommons.org/licenses/by-sa/4.0/\nOriginal project: https://scienceqa.github.io/\nImage subsets and manifests retain attribution; downloaded images are not committed to this course.\nID: natural science grades 1–6; OOD: natural science grades 7–12; general: other subjects. This is an operational shift, not a claim of universal model ability.\n')
    return manifest


## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Предпочтения из визуальных ответов

*Вес: 1 балл.*

**Постановка.** Подготовьте ScienceQA через открытый загрузчик. Для каждого изображения chosen — правильная буква, rejected — следующий неверный вариант. Явно пометьте: это предпочтения на основе эталонной разметки, а не экспертные оценки полезности.

**Результат.** Список пар с provenance.

**Критерий принятия.** Изображения существуют, ответы различаются и соответствуют вариантам; лицензия и источник сохранены.


In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: непустые различающиеся ответы на существующих изображениях
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Отложенный тест и смещения пар

*Вес: 1 балл.*

**Постановка.** Используйте official train для DPO и ID test для итоговой оценки. Сверьте ID и хеши изображений. Посчитайте распределение типов ошибок и долю случаев, где chosen длиннее rejected.

**Результат.** preferences.jsonl и pair_audit.json.

**Критерий принятия.** Нет пересечений; длина ответа не используется как скрытый эталон правильности.


In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/preferences.jsonl и pair_audit.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Начальная и фиксированная политики

*Вес: 1 балл.*

**Постановка.** Создайте обучаемый адаптер и отдельный фиксированный reference. Используйте адаптер работы 09, если он доступен; иначе обе политики начинаются с опубликованной instruction-tuned модели. Запишите выбранный источник.

**Результат.** reference_policy.json.

**Критерий принятия.** Начальные политики совпадают; параметры reference не обновляются; запуск без адаптера 09 не называется продолжением собственного SFT.


In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: SFT-адаптер и отдельная замороженная reference policy
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Log-prob мультимодального ответа

*Вес: 1 балл.*

**Постановка.** Реализуйте prepare_answer и completion_logprob. Сохраните все image tensors процессора, исключите prompt из суммы логарифмов вероятностей и учитывайте только токены ответа и окончания.

**Результат.** Функции подготовки и scoring.

**Критерий принятия.** Префикс токенов full совпадает с отдельно токенизированным prompt; выбран правильный сдвиг next-token labels; score дифференцируем по policy.


In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: мультимодальный Dataset с изображениями и парными ответами
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Целевая функция DPO

*Вес: 1 балл.*

**Постановка.** Настройте beta=0.1, AdamW lr=5e-6, accumulation=8 и 40 optimizer steps. Реализуйте отрицательный log-sigmoid разности policy/ref margins; в optimizer включите только trainable adapter.

**Результат.** dpo_protocol.json и optimizer.

**Критерий принятия.** Рост policy-margin при фиксированном reference уменьшает loss; вычисление reference идёт без градиентов.


In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: DPOTrainer с обучаемым и эталонным адаптерами
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Обучение предпочтениям

*Вес: 1 балл.*

**Постановка.** Выполните реальное DPO-обучение, сохраняя loss каждого шага. Не маскируйте NaN. Сохраните обучаемый LoRA в отдельный каталог.

**Результат.** aligned_adapter/train и dpo_metrics.json.

**Критерий принятия.** Обучение выполнило заданный бюджет; loss конечен, адаптер сохранён; smoke не считается полным экспериментом.


In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: aligned_adapter и dpo_metrics.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Исходные ответы на тесте

*Вес: 1 балл.*

**Постановка.** Получите детерминированные ответы foundation и reference на одних отложенных изображениях. Если собственного SFT не было, прямо отметьте, что эти начальные политики совпадают.

**Результат.** Парные исходные ответы.

**Критерий принятия.** Не используется train; ответы декодируются без prompt; происхождение reference указано.


In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: парные ответы первых двух политик
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Ответы после DPO

*Вес: 1 балл.*

**Постановка.** Активируйте обученный адаптер и получите ответы для тех же ID. Сохраните foundation/reference/aligned вместе с правильной буквой.

**Результат.** blind_predictions.jsonl.

**Критерий принятия.** Каждый ID имеет три фактических ответа; порядок выборок не используется вместо ID.


In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/blind_predictions.jsonl для трёх политик
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Автоматическая и ручная оценка

*Вес: 1 балл.*

**Постановка.** Рассчитайте EM по эталонным буквам. Создайте форму ручной проверки предпочтений, галлюцинаций и формата; заполните её самостоятельно после просмотра ответов. Пропуски обозначайте pending, а не нулём.

**Результат.** objective_scores.json и blind_judgements.jsonl.

**Критерий принятия.** EM не назван человеческим win rate; неопределённые суждения не включены в ручные метрики; для финальной сдачи форма заполнена.


In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: проверенный blind_judgements.jsonl на каждый id
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Эффект DPO

*Вес: 1 балл.*

**Постановка.** Сопоставьте EM трёх политик, а по заполненным суждениям — win rate, hallucination rate и соблюдение формата. Укажите число оценённых записей и ограничения label-derived предпочтений.

**Результат.** blind_report.json и вывод.

**Критерий принятия.** Метрики имеют реальные знаменатели; pending-оценка не выдаётся за завершённую; вывод не обобщает предпочтение правильной буквы на безопасность или универсальную полезность.


In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: blind_report.json из экспертных оценок и ответов
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [ ]:
# Реализуйте творческое мини-исследование по постановке выше.
raise NotImplementedError("Реализуйте бонусное исследование")

## Анализ результатов

Сопоставьте измеренные показатели на фиксированной отложенной выборке, укажите ограничения выборки и разберите ошибки по сохранённым предсказаниям. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.